# CV4CHL - Final Submission Assembly (optional interactive wrapper)

> **This notebook is NOT executed by `reproduce.py`.** The deterministic
> assembly pipeline is run directly via the modules under `src/cv4chl/`
> (`ml_merge` → `clinical_consistency` → `final_assembly`); see
> `reproduce.py` and `docs/retraining.md`.
>
> This notebook is a thin interactive wrapper around the same modules,
> useful when a reviewer wants to step through the assembly stages cell by
> cell in Jupyter. Running it is optional and produces the same artifacts
> as `python reproduce.py --skip-train`.

Thin notebook wrapper around `src/cv4chl/final_assembly.py`.

In [ ]:
# === Cell 1: Setup & Imports ===
import hashlib
import os
import pandas as pd
import sys
import time
from pathlib import Path

_cell_times = {}
_cell_start = None

def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'> {name}')

def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'OK {name} - {elapsed:.1f}s')

def find_repo_root():
    env_root = os.environ.get('CV4CHL_REPO_ROOT')
    if env_root:
        return Path(env_root).resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'src' / 'cv4chl').exists() and (candidate / 'submissions').exists():
            return candidate
    return here

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cv4chl.clinical_consistency import write_clinical_consistency
from cv4chl.final_assembly import FINAL_NAME, write_final
from cv4chl.ml_merge import RUNTIME_ML_SOURCE_FILES, write_preclinical_ml_merge
print(f'REPO_ROOT: {REPO_ROOT}')


In [ ]:
# === Cell 2: Write Preclinical ML Merge ===
cell_start('Cell 2: Write Preclinical ML Merge')
runtime_sources_present = any((REPO_ROOT / p).exists() for p in RUNTIME_ML_SOURCE_FILES.values())
preclinical_source = write_preclinical_ml_merge(
    REPO_ROOT,
    REPO_ROOT / '5_outputs' / 'submissions' / 'preclinical_ml_merge_track1.csv',
    use_runtime=runtime_sources_present,
)
print(f'Preclinical ML merge written: {preclinical_source}')
print(f'Runtime model sources present: {runtime_sources_present}')
cell_end('Cell 2: Write Preclinical ML Merge')


In [ ]:
# === Cell 3: Generate Clinical Consistency Source ===
cell_start('Cell 3: Generate Clinical Consistency Source')
clinical_source, clinical_log = write_clinical_consistency(
    REPO_ROOT,
    REPO_ROOT / '5_outputs' / 'submissions' / 'clinical_consistency_track1.csv',
    use_runtime=True,
)
print(f'Clinical consistency source written: {clinical_source}')
print(f'Clinical second-pass cells: {len(pd.read_csv(clinical_log))}')
cell_end('Cell 3: Generate Clinical Consistency Source')


In [ ]:
# === Cell 4: Assemble Retrained CSV ===
cell_start('Cell 4: Assemble Retrained CSV')
out_path = write_final(
    REPO_ROOT,
    REPO_ROOT / '5_outputs' / 'submissions' / 'retrained_final.csv',
    clinical_source_path=clinical_source,
)
h = hashlib.sha256(out_path.read_bytes()).hexdigest()
print(f'Retrained CSV written: {out_path}')
print(f'sha256: {h}')
cell_end('Cell 4: Assemble Retrained CSV')


In [ ]:
# === Cell 5: Summary ===
cell_start('Cell 5: Summary')
print('Preclinical stage: src/cv4chl/ml_merge.py')
print('Clinical stage: src/cv4chl/clinical_consistency.py')
print('Assembly module: src/cv4chl/final_assembly.py')
print(f'Preclinical source: {preclinical_source.relative_to(REPO_ROOT)}')
print(f'Clinical source: {clinical_source.relative_to(REPO_ROOT)}')
print(f'Clinical log: {clinical_log.relative_to(REPO_ROOT)}')
print('Output: 5_outputs/submissions/retrained_final.csv')
print(f'Total time: {sum(t for t in _cell_times.values() if t is not None):.1f}s')
cell_end('Cell 5: Summary')
